In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import math
# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list the files in the input directory

import os
print(os.listdir("../input"))

# Any results you write to the current directory are saved as output.


['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']


In [2]:
# Set columns to most suitable type to optimize for memory usage
types = {'fare_amount': 'float32',
         'pickup_longitude': 'float32',
         'pickup_latitude': 'float32',
         'dropoff_longitude': 'float32',
         'dropoff_latitude': 'float32',
         'passenger_count': 'uint8'}


# Columns to keep (basically discarding the 'key' column) - thanks to the suggestion by mhviraf
cols = ['fare_amount', 'pickup_datetime', 'pickup_longitude', 'pickup_latitude', 
        'dropoff_longitude', 'dropoff_latitude', 'passenger_count']


In [3]:
# train = pd.read_csv('../input/train.csv', nrows=5000000, usecols=cols, dtype=types)
train = pd.read_csv('../input/train.csv', nrows=1000000, usecols=cols, dtype=types)
test = pd.read_csv('../input/test.csv')
samp = pd.read_csv('../input/sample_submission.csv')


In [4]:
train.dropna(how = 'any', axis = 'rows', inplace=True)
train = train[train.fare_amount > 0]
train = train[train['passenger_count'] <= 6]


In [5]:
latitude_mask_pickup = (train.pickup_latitude > -90) &  (train.pickup_latitude < 90)

train = train[latitude_mask_pickup]

latitude_mask_dropoff = (train.dropoff_latitude > -90) &  (train.dropoff_latitude < 90)

train = train[latitude_mask_dropoff]


In [6]:
longitude_mask_pickup = (train.pickup_longitude > -180) &  (train.pickup_longitude < 180)

train = train[longitude_mask_pickup]

longitude_mask_dropoff = (train.dropoff_longitude > -180) &  (train.dropoff_longitude < 180)

train = train[longitude_mask_dropoff]


In [7]:
all_data = pd.concat((train, test)).reset_index(drop=True)

all_data.drop(['fare_amount'] , axis=1, inplace=True)
y = train.fare_amount.values
n_train = len(train)
n_test = len(test)
test_id = test.key


In [8]:
def week_num(day):
    '''
    given the day of the month, return the week number of the month
    '''
    if day <=7 : return 'first'
    if (day > 7) and (day <= 14): return 'second'
    if (day > 14) and (day <= 21): return 'third'
    if (day > 21) and (day <= 28): return 'fourth'
    return 'fifth'


In [9]:
def add_time_features(data):
    data.pickup_datetime =  pd.to_datetime(data.pickup_datetime)
    # Date Features 

    data['hour'] = data.pickup_datetime.dt.hour
    data['day_of_week'] = data.pickup_datetime.dt.weekday_name
    data['day_of_month'] = data.pickup_datetime.dt.day
    data['week_of_month'] = data.day_of_month.map(week_num)
    data['month'] = data.pickup_datetime.dt.month
    data['year'] = data.pickup_datetime.dt.year

    data.hour = data.hour.astype(str)
    data.month = data.month.astype(str)
    data.year = data.year.astype(str)
    data.drop('day_of_month', axis=1, inplace=True)
    
    return data


In [10]:
def add_geo_features(data):
    # Geo Features 
    data['abs_diff_longitude'] = (data.dropoff_longitude - data.pickup_longitude).abs()
    data['abs_diff_latitude'] = (data.dropoff_latitude - data.pickup_latitude).abs()

    data['manhattan_distance'] = data['abs_diff_longitude'] + data['abs_diff_latitude']

    data['squared_long'] = np.power(data['abs_diff_longitude'],2)
    data['squared_lat'] = np.power(data['abs_diff_latitude'],2)

    data['euclid_distance'] = np.sqrt(data['squared_long'] + data['squared_lat'])
    
    return data


In [11]:
all_data = add_time_features(all_data)


AttributeError: 'DatetimeProperties' object has no attribute 'weekday_name'

In [12]:
all_data = add_geo_features(all_data)


In [13]:
features = ['passenger_count', 'hour', 'day_of_week', 'week_of_month', 'month', 'year', 'abs_diff_longitude', 'abs_diff_latitude', 'manhattan_distance', 'euclid_distance']

all_data = all_data[features]

all_data = pd.get_dummies(all_data)


KeyError: "['day_of_week', 'week_of_month', 'month', 'year'] not in index"

In [14]:
x = all_data[:n_train]
x_test = all_data[n_train:]


In [15]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
model_1 = RandomForestRegressor()


In [16]:
model_1.fit(x, y)

model_1_pred = model_1.predict(x_test)
sub_1 = pd.DataFrame()
sub_1['key'] = test_id
sub_1['fare_amount'] = model_1_pred
sub_1.to_csv('submission_rf.csv', index=False)


TypeError: float() argument must be a string or a real number, not 'Timestamp'

In [17]:
from keras.models import Sequential
from keras.layers import Dense


2026-01-07 05:26:19.317426: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767763579.328229      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767763579.331702      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 05:26:19.346136: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [18]:
num_features = len(x.columns)


In [19]:
model = Sequential()
model.add(Dense(30, input_dim=num_features, activation='relu'))
model.add(Dense(15, activation='relu'))
model.add(Dense(7, activation='relu'))
model.add(Dense(3, activation='relu'))
model.add(Dense(1, activation='linear'))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-01-07 05:26:24.941867: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1767763584.942315      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:46:00.0, compute capability: 8.6


In [20]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 30)             │           450 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 15)             │           465 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            24 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,055 (4.12 KB)

 Trainable params: 1,055 (4.12 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
model.compile(loss='mean_squared_error', optimizer='adam')


In [22]:
model.fit(x, y)


ValueError: Invalid dtype: datetime64[ns, UTC]

In [23]:
test_pred = model.predict(x_test)


ValueError: Invalid dtype: datetime64[ns, UTC]

In [24]:
sub = pd.DataFrame()
sub['key'] = test_id
sub['fare_amount'] = test_pred
sub.to_csv('submission_nn.csv', index=False)


NameError: name 'test_pred' is not defined